# Extraction batch de PDFs → Parquet

Ce notebook traite **~30 000 PDFs** (~25 pages chacun) en parallèle et écrit les résultats dans des fichiers **Parquet partitionnés**.  
Il gère :
- La **parallélisation** via `ProcessPoolExecutor`
- L'**écriture incrémentale** (un fichier `.parquet` par batch → jamais tout en RAM)
- La **reprise sur interruption** via un checkpoint (les PDFs déjà traités sont skippés)
- Un **log d'erreurs** en JSONL

---
### Structure des sorties
```
outputs/
  parquet/
    batch_000000.parquet
    batch_000001.parquet
    ...
  errors.jsonl
  checkpoint.json
```

**Lire l'ensemble du dataset après traitement :**
```python
import pandas as pd
df_all = pd.read_parquet("outputs/parquet/")
```

In [ ]:
# ===========================================================
#  0. DÉPENDANCES  (à lancer une seule fois)
# ===========================================================
# !pip install pdfplumber pandas pyarrow tqdm

In [ ]:
# ===========================================================
#  1. IMPORTS
# ===========================================================
import sys
import os
import json
import logging
from pathlib import Path
from datetime import datetime
from concurrent.futures import ProcessPoolExecutor, as_completed

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.notebook import tqdm

# Ajouter le répertoire contenant analyse.py au path Python
PROJECT_DIR = Path("__file__").resolve().parent
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("Imports OK")

In [ ]:
# ===========================================================
#  2. CONFIGURATION  ← adapter ici
# ===========================================================

# Dossier contenant tous vos PDFs (cherche récursivement)
INPUT_DIR = Path("/chemin/vers/vos/pdfs")  # ← À MODIFIER

# Dossiers de sortie (créés automatiquement)
OUTPUT_DIR    = PROJECT_DIR / "outputs" / "parquet"
ERROR_LOG     = PROJECT_DIR / "outputs" / "errors.jsonl"
CHECKPOINT    = PROJECT_DIR / "outputs" / "checkpoint.json"

# Nombre de PDFs à regrouper par fichier Parquet
# 500 PDFs × ~25 lignes/page × 25 pages ≈ 300 000 lignes/batch → ~10-20 Mo/fichier
BATCH_SIZE = 500

# Nombre de workers parallèles
# Règle : os.cpu_count() // 2 est un bon point de départ pour éviter la saturation I/O
MAX_WORKERS = max(1, (os.cpu_count() or 4) // 2)

print(f"INPUT_DIR  : {INPUT_DIR}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")
print(f"BATCH_SIZE : {BATCH_SIZE}")
print(f"MAX_WORKERS: {MAX_WORKERS}")

In [ ]:
# ===========================================================
#  3. FONCTIONS UTILITAIRES
# ===========================================================

def setup_output_dirs():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    ERROR_LOG.parent.mkdir(parents=True, exist_ok=True)

def load_checkpoint() -> set:
    """Charge l'ensemble des noms de fichiers déjà traités."""
    if CHECKPOINT.exists():
        with open(CHECKPOINT, "r", encoding="utf-8") as f:
            return set(json.load(f))
    return set()

def save_checkpoint(done: set):
    """Persiste le checkpoint après chaque batch."""
    with open(CHECKPOINT, "w", encoding="utf-8") as f:
        json.dump(list(done), f, ensure_ascii=False)

def append_errors(errors: list):
    """Ajoute des erreurs au log JSONL."""
    with open(ERROR_LOG, "a", encoding="utf-8") as f:
        for e in errors:
            f.write(json.dumps(e, ensure_ascii=False) + "\n")

def collect_pdf_paths(directory: Path) -> list:
    """Retourne la liste triée de tous les .pdf dans le dossier (récursif)."""
    paths = sorted(directory.rglob("*.pdf"))
    print(f"{len(paths)} fichiers PDF trouvés dans {directory}")
    return paths

def batched(lst: list, n: int):
    """Découpe une liste en sous-listes de taille n."""
    for i in range(0, len(lst), n):
        yield lst[i : i + n]

print("Utilitaires OK")

In [ ]:
# ===========================================================
#  4. FONCTION WORKER  (doit être importable par ProcessPoolExecutor)
# ===========================================================
# NOTE: les imports sont DANS la fonction pour garantir la picklabilité
# sur macOS (spawn context) et Windows.

def extract_one_pdf(pdf_path_str: str):
    """
    Extrait un PDF et retourne (pdf_filename, df_or_None, error_msg_or_None).
    Cette fonction est exécutée dans un processus fils.
    """
    import sys
    from pathlib import Path
    import pandas as pd

    # S'assurer que le projet est dans le path du processus fils
    project = str(Path(pdf_path_str).resolve().parent.parent)  # fallback
    # Le chemin exact du projet est injecté via l'initializer (voir cell 5)
    try:
        from analyse import PdfTwoColumnExtractor
    except ImportError:
        # Fallback : essayer de trouver analyse.py depuis le PATH injecté
        return Path(pdf_path_str).name, None, "ImportError: analyse.py introuvable dans le PATH"

    path = Path(pdf_path_str)
    try:
        extractor = PdfTwoColumnExtractor()
        df = extractor.to_dataframe(pdf_path_str)
        if df.empty:
            return path.name, None, "DataFrame vide après extraction"
        df.insert(0, "pdf_file", path.name)
        return path.name, df, None
    except Exception as exc:
        return path.name, None, f"{type(exc).__name__}: {exc}"


def initializer(project_path: str):
    """Injecté dans chaque processus fils pour configurer le sys.path."""
    import sys
    if project_path not in sys.path:
        sys.path.insert(0, project_path)

print("Worker OK")

In [ ]:
# ===========================================================
#  5. BOUCLE PRINCIPALE D'EXTRACTION
# ===========================================================

setup_output_dirs()

# 5a. Collecter tous les PDFs
all_pdfs = collect_pdf_paths(INPUT_DIR)

# 5b. Charger le checkpoint (PDFs déjà traités)
done_set = load_checkpoint()
todo_pdfs = [p for p in all_pdfs if p.name not in done_set]
print(f"À traiter : {len(todo_pdfs)} / {len(all_pdfs)} ({len(done_set)} déjà faits)")

# 5c. Statistiques globales
total_rows   = 0
total_errors = 0
batch_index  = len(list(OUTPUT_DIR.glob("batch_*.parquet")))  # continuer la numérotation

# Schéma PyArrow fixe pour la cohérence entre tous les fichiers Parquet
SCHEMA = pa.schema([
    pa.field("pdf_file",    pa.string()),
    pa.field("titre",       pa.string()),
    pa.field("description", pa.string()),
])

# 5d. Traitement par batches
batches = list(batched(todo_pdfs, BATCH_SIZE))
print(f"{len(batches)} batches de {BATCH_SIZE} PDFs")

with tqdm(total=len(todo_pdfs), desc="PDFs traités", unit="pdf") as pbar:
    for batch in batches:
        batch_dfs   = []
        batch_errors = []

        with ProcessPoolExecutor(
            max_workers=MAX_WORKERS,
            initializer=initializer,
            initargs=(str(PROJECT_DIR),),
        ) as executor:
            futures = {
                executor.submit(extract_one_pdf, str(p)): p
                for p in batch
            }
            for future in as_completed(futures):
                pdf_name, df, error = future.result()
                pbar.update(1)
                pbar.set_postfix(errors=total_errors, rows=total_rows)

                if error:
                    total_errors += 1
                    batch_errors.append({
                        "pdf_file":  pdf_name,
                        "error":     error,
                        "timestamp": datetime.utcnow().isoformat() + "Z",
                    })
                else:
                    total_rows += len(df)
                    batch_dfs.append(df)
                    done_set.add(pdf_name)

        # Écrire le batch en Parquet
        if batch_dfs:
            batch_df = pd.concat(batch_dfs, ignore_index=True)
            # Garantir les types string (évite les NaN de type object)
            for col in ["titre", "description"]:
                batch_df[col] = batch_df[col].fillna("").astype(str)
            table = pa.Table.from_pandas(batch_df, schema=SCHEMA, preserve_index=False)
            out_path = OUTPUT_DIR / f"batch_{batch_index:06d}.parquet"
            pq.write_table(table, out_path, compression="snappy")
            batch_index += 1

        # Persister les erreurs et le checkpoint après chaque batch
        if batch_errors:
            append_errors(batch_errors)
        save_checkpoint(done_set)

print(f"\n✓ Terminé — {total_rows:,} lignes extraites, {total_errors} erreurs")

In [ ]:
# ===========================================================
#  6. VÉRIFICATION RAPIDE
# ===========================================================

parquet_files = sorted(OUTPUT_DIR.glob("batch_*.parquet"))
print(f"{len(parquet_files)} fichiers Parquet dans {OUTPUT_DIR}")
for f in parquet_files:
    size_mb = f.stat().st_size / 1e6
    meta = pq.read_metadata(f)
    print(f"  {f.name}  →  {meta.num_rows:,} lignes  ({size_mb:.1f} Mo)")

In [ ]:
# ===========================================================
#  7. LECTURE DU DATASET COMPLET
# ===========================================================
# Tous les fichiers batch_*.parquet sont lus en un seul appel.
# PyArrow fusionne les fichiers sans les charger tous en RAM d'un coup.

df_all = pd.read_parquet(OUTPUT_DIR)
print(f"Dataset complet : {len(df_all):,} lignes × {df_all.shape[1]} colonnes")
print(f"PDFs uniques    : {df_all['pdf_file'].nunique():,}")
print("\nEchantillon :")
df_all.sample(min(10, len(df_all)))

In [ ]:
# ===========================================================
#  8. CONSOLIDATION OPTIONNELLE EN UN SEUL FICHIER PARQUET
# ===========================================================
# Utile si vous voulez un fichier unique à transmettre/déposer en base.
# Attention : nécessite que tout tienne en RAM.

SINGLE_OUTPUT = PROJECT_DIR / "outputs" / "all_extractions.parquet"

df_all = pd.read_parquet(OUTPUT_DIR)
df_all.to_parquet(SINGLE_OUTPUT, index=False, compression="snappy")
size_mb = SINGLE_OUTPUT.stat().st_size / 1e6
print(f"Fichier consolidé : {SINGLE_OUTPUT}  ({size_mb:.1f} Mo)")

In [ ]:
# ===========================================================
#  9. INSPECTION DES ERREURS
# ===========================================================

if ERROR_LOG.exists():
    errors_df = pd.read_json(ERROR_LOG, lines=True)
    print(f"{len(errors_df)} erreurs enregistrées")
    print("\nTop erreurs :")
    print(errors_df["error"].value_counts().head(10))
    display(errors_df.head(20))
else:
    print("Aucune erreur enregistrée.")